## 1. Agglomerative clustering and label assignment

Agglomerative clustering starts with one cluster per attribute and merges clusters according to average pairwise distance. It accepts distances, not similarities, so the blended values are converted with `distance = 1 - similarity`. `n_clusters=None` means the number of groups is not fixed in advance; `distance_threshold = 0.70` controls when merging stops. A lower threshold generally produces more, smaller groups. `labels[i]` is the cluster assigned to `attributes[i]`; the numeric label itself has no semantic ranking.

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE

THRESHOLD = 0.70
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

distance_matrix = 1 - similarity
model = AgglomerativeClustering(
    n_clusters=None,
    metric="precomputed",
    linkage="average",
    distance_threshold=THRESHOLD,
)
labels = model.fit(distance_matrix).labels_

sizes = Counter(labels)
multi = [label for label, size in sizes.items() if size > 1]
print(f"{len(sizes)} clusters: {len(multi)} with multiple attributes, {len(sizes) - len(multi)} singletons")

# Print members so each label can be inspected before the visualization.
members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in sizes
}
for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} attributes)")
        for member in members:
            print(f"  - {member}")

138 clusters: 85 with multiple attributes, 53 singletons

Cluster 14 (20 attributes)
  - Highly breathable
  - Breathable mesh back
  - Insulating & breathable
  - Breathable
  - Breathable, windproof fabric
  - Cool and comfortable
  - Breathable construction
  - Extremely breathable
  - Super breathable
  - Ideal for hot conditions
  - Thermal insulation with exceptional breathability
  - Insulating & Breathable
  - Breathable mesh inserts
  - Breathable insulated backhand
  - Extremely breathable 40K (40,000 g/m²/24h)
  - Maximum breathability
  - Breathable 15K
  - Breathable mesh back panel
  - Breathable fabric
  - Ultralight and breathable

Cluster 0 (15 attributes)
  - Secure hold with elasticated grippers
  - Secure silicone gripper hem
  - Elastic silicone grippers
  - Silicone palm grip
  - Silicone gripper
  - InsideGrip™ technology
  - Silicone-enhanced palm sections
  - Performance grip
  - Secure grip and control
  - Silicone fingertips
  - Silicone Hem Gripper
  - Silic

## Clustering evaluation

_Reserved for evaluation later. No evaluation metrics or methods are calculated in this notebook._

## 2. Cluster visualization

t-SNE projects the same precomputed distances to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity in this projection, but the plot can distort some distances. The largest multi-attribute groups receive distinct colors, other groups are muted, and singleton clusters are shown as hollow markers. Hover over a point to see its attribute and group. The fixed perplexity of 30 requires more than 30 input attributes.

In [18]:
import importlib
import plotly.io._renderers as plotly_renderers

importlib.reload(plotly_renderers)

<module 'plotly.io._renderers' from 'c:\\Users\\Francesco\\Documents\\CompTools\\CTFDS-Project\\.venv\\Lib\\site-packages\\plotly\\io\\_renderers.py'>

In [19]:
from IPython.display import HTML, display

HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar attribute as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups]
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1])

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Attribute clusters: context-aware similarity (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))